In [ ]:
"""
Synthetic Source Attribution — End-to-End PyTorch GPU/CPU Pipeline
=========================================================================
Pipeline stages (in order):
  1. System config, globals, imports, seeding
  2. Data loading & validation
  3. Data preprocessing (normalisation, fold splitting)
  4. Model training with per-epoch train + val metrics (CV, no leakage)
  5. Inference / test-set prediction (ensemble)
  6. Submission & outputs
""";

# STAGE 0 - CLEANING

In [ ]:
import shutil
shutil.rmtree("/kaggle/working", ignore_errors=True)

# STAGE 1 - SYSTEM CONFIG, GLOBALS, IMPORTS, SEEDING

## IMPORTS

In [ ]:
from __future__ import annotations

import os
import gc
import json
import copy
import logging
import math
import random
import shutil
import time
from pathlib import Path
from typing import Sequence

import numpy as np
import pandas as pd
import psutil
from PIL import Image, ImageFile, ImageOps
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torch.amp import autocast, GradScaler

import timm

ImageFile.LOAD_TRUNCATED_IMAGES = True

print("Imports ready")

## CONTROL CONFIG

In [ ]:
# ── Control panel ─────────────────────────────────────────────────────────────
CONTROL_PANEL: dict[str, dict] = {
    "training": {
        "seed": 42,
        "num_classes": 10,
        "num_folds": 1,
        "num_epochs": 30,

        "checkpoint_selection_metric": "generalization_score",

        "early_stop_patience": 3,
        "checkpoint_keep_top_k": 4,

        "use_lr_plateau": True,
        "plateau_patience": 3,
        "plateau_factor": 0.5,
        "plateau_min_lr": 1e-7,

        "use_amp": True,
        "use_compile": False,

        "grad_accum_steps": 1,

        "deterministic": False,
    },

    "regularization_defaults": {
        "weight_decay": 1e-4,
        "label_smoothing": 0.09,

        "mixup_alpha": 0.0,
        "mixup_prob": 0.0,

        "cutmix_alpha": 0.0,
        "cutmix_prob": 0.0,

        "dropout": 0.20,
        "grad_clip_norm": 1.0,
    },

    "blend": {
        "mode": "stacking",
        "weight_metric": "selection_value",
        "stacking_learner": "logreg",

        "use_tta": False,
        "tta_n": 4,
    },

    "monitoring": {
        "verbose_training": True,
        "train_eval_each_epoch": False,
        "print_cuda_memory_every": 0,
    },

    "generalization": {
        "val_weight": 1.0,
        "low_train_reward": 0.15,
        "overfit_penalty": 2.0,
        "balance_penalty": 0.5,
    },

    "resources": {
        "session_budget_secs": 8.5 * 3600,
        "disk_limit_gib": 17.0,
        "ram_warn_gib": 26.0,
    },

    "runtime": {
        "run_training": True,
        "run_inference_only": False,

        "inference_only_path":
            "/kaggle/input/models/punyakdei/pipe-1/pytorch/default/1",
    },

    "extra_verbose": {
        "print_model": False
    }
}


# ── Derived config ────────────────────────────────────────────────────────────
TRAIN_CFG = CONTROL_PANEL["training"]
BLEND_CFG = CONTROL_PANEL["blend"]
MONITOR_CFG = CONTROL_PANEL["monitoring"]
RESOURCE_CFG = CONTROL_PANEL["resources"]
RUNTIME_CFG = CONTROL_PANEL["runtime"]
EXTRA_VERBOSE = CONTROL_PANEL["extra_verbose"]

SEED = int(TRAIN_CFG["seed"])

NUM_CLASSES = int(TRAIN_CFG["num_classes"])
NUM_FOLDS = int(TRAIN_CFG["num_folds"])
NUM_EPOCHS = int(TRAIN_CFG["num_epochs"])

DEVICE_TYPE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
USE_AMP = DEVICE_TYPE == "cuda" and bool(TRAIN_CFG["use_amp"])
USE_COMPILE = bool(TRAIN_CFG["use_compile"])

GRAD_ACCUM_STEPS = int(TRAIN_CFG["grad_accum_steps"])
DETERMINISTIC = bool(
    TRAIN_CFG["deterministic"]
)

NUM_WORKERS = min(4, os.cpu_count() or 2) if DEVICE_TYPE == "cuda" else 0
PIN_MEMORY = DEVICE_TYPE == "cuda"

TRAIN_EVAL_EACH_EPOCH = bool(
    MONITOR_CFG["train_eval_each_epoch"]
)

VERBOSE_TRAINING = bool(
    MONITOR_CFG["verbose_training"]
)

PRINT_CUDA_MEMORY_EVERY = int(
    MONITOR_CFG["print_cuda_memory_every"]
)

USE_LR_PLATEAU = bool(
    TRAIN_CFG["use_lr_plateau"]
)

PLATEAU_PATIENCE = int(
    TRAIN_CFG["plateau_patience"]
)

PLATEAU_FACTOR = float(
    TRAIN_CFG["plateau_factor"]
)

PLATEAU_MIN_LR = float(
    TRAIN_CFG["plateau_min_lr"]
)

CHECKPOINT_SELECTION_METRIC = str(
    TRAIN_CFG["checkpoint_selection_metric"]
)

EARLY_STOP_PATIENCE = int(
    TRAIN_CFG["early_stop_patience"]
)

CHECKPOINT_KEEP_TOP_K = int(
    TRAIN_CFG["checkpoint_keep_top_k"]
)

USE_TTA = bool(
    BLEND_CFG["use_tta"]
)

TTA_N = int(
    BLEND_CFG["tta_n"]
)

BLEND_MODE = str(
    BLEND_CFG["mode"]
)

STACKING_LEARNER = str(
    BLEND_CFG["stacking_learner"]
)

BLEND_WEIGHT_METRIC = str(
    BLEND_CFG["weight_metric"]
)

SESSION_START_TIME = time.time()

SESSION_BUDGET_SECS = float(
    RESOURCE_CFG["session_budget_secs"]
)

DISK_LIMIT_GIB = float(
    RESOURCE_CFG["disk_limit_gib"]
)

RAM_WARN_GIB = float(
    RESOURCE_CFG["ram_warn_gib"]
)

RUN_TRAINING = bool(
    RUNTIME_CFG["run_training"]
)

RUN_INFERENCE_ONLY = bool(
    RUNTIME_CFG["run_inference_only"]
)

INFERENCE_ONLY_PATH = str(
    RUNTIME_CFG["inference_only_path"]
)

PRINT_MODEL = bool(
    EXTRA_VERBOSE["print_model"]
)

## DEVICE

In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DEVICE_TYPE = DEVICE.type

NUM_DEVICES = (
    torch.cuda.device_count()
    if DEVICE_TYPE == "cuda"
    else 0
)

USE_AMP = (
    DEVICE_TYPE == "cuda"
    and bool(TRAIN_CFG["use_amp"])
)

AMP_DTYPE = (
    torch.bfloat16
    if DEVICE_TYPE == "cuda"
    and torch.cuda.is_bf16_supported()
    else torch.float16
)

if DEVICE_TYPE == "cuda":

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

## PATH

In [ ]:
# ── Path helpers ──────────────────────────────────────────────────────────────
def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p


# ── Base paths ────────────────────────────────────────────────────────────────
COMPETITION_ROOT = Path(
    os.environ.get(
        "COMPETITION_ROOT",
        "/kaggle/input/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge"
    )
)

DATA_ROOT = COMPETITION_ROOT / "Data" / "Data"

TRAIN_CSV = DATA_ROOT / "training.csv"
TEST_CSV  = DATA_ROOT / "test.csv"

TRAIN_DIR = DATA_ROOT / "Training"
TEST_DIR  = DATA_ROOT / "Test"

WORKING_ROOT = ensure_dir(
    Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
)


# ── Run isolation ─────────────────────────────────────────────────────────────
RUN_NAME = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR  = ensure_dir(WORKING_ROOT / RUN_NAME)


# ── Artifact directories ──────────────────────────────────────────────────────
ARTIFACTS_DIR  = ensure_dir(RUN_DIR / "artifacts")

PROCESSED_DIR  = ensure_dir(ARTIFACTS_DIR / "processed")
CHECKPOINT_DIR = ensure_dir(ARTIFACTS_DIR / "checkpoints")
FINAL_DIR      = ensure_dir(ARTIFACTS_DIR / "final_models")
LOG_DIR        = ensure_dir(ARTIFACTS_DIR / "logs")

OUTPUT_DIR     = ensure_dir(ARTIFACTS_DIR / "outputs")
INFERENCE_DIR  = ensure_dir(OUTPUT_DIR / "inference")
VALIDATION_DIR = ensure_dir(OUTPUT_DIR / "validation")

CACHE_DIR      = ensure_dir(ARTIFACTS_DIR / "cache")


# ── Torch/timm cache ──────────────────────────────────────────────────────────
os.environ["TORCH_HOME"] = str(CACHE_DIR)

## CHECKPOINTS OF CAFORMER MODELS (PRETRAINED)

In [ ]:
ALL_CAFORMER_CHECKPOINTS = [

    # =========================================================================
    # CAFormer-S18
    # Small lightweight variant (~26M params), good speed/accuracy balance.
    # =========================================================================

    # Trained directly on ImageNet-1K at 224x224 resolution.
    "caformer_s18.sail_in1k",

    # Same S18 model but trained/evaluated at higher 384x384 resolution.
    "caformer_s18.sail_in1k_384",

    # Pretrained on larger ImageNet-22K dataset for better feature learning.
    "caformer_s18.sail_in22k",

    # ImageNet-22K pretrained model fine-tuned on ImageNet-1K.
    "caformer_s18.sail_in22k_ft_in1k",

    # 22K pretrained + 1K fine-tuned version at 384x384 resolution.
    "caformer_s18.sail_in22k_ft_in1k_384",


    # =========================================================================
    # CAFormer-S36
    # Larger small-model variant (~39M params), better accuracy than S18.
    # =========================================================================

    # Standard ImageNet-1K pretrained checkpoint (224x224).
    "caformer_s36.sail_in1k",

    # Higher-resolution 384x384 ImageNet-1K trained checkpoint.
    "caformer_s36.sail_in1k_384",

    # Pretrained on ImageNet-22K for stronger representations.
    "caformer_s36.sail_in22k",

    # 22K pretrained then fine-tuned on ImageNet-1K.
    "caformer_s36.sail_in22k_ft_in1k",

    # Fine-tuned high-resolution 384x384 variant.
    "caformer_s36.sail_in22k_ft_in1k_384",


    # =========================================================================
    # CAFormer-M36
    # Medium-sized variant (~56M params), strong accuracy/performance tradeoff.
    # =========================================================================

    # Standard 224x224 ImageNet-1K pretrained model.
    "caformer_m36.sail_in1k",

    # High-resolution 384x384 ImageNet-1K version.
    "caformer_m36.sail_in1k_384",

    # Pretrained using ImageNet-22K dataset.
    "caformer_m36.sail_in22k",

    # 22K pretraining followed by 1K fine-tuning.
    "caformer_m36.sail_in22k_ft_in1k",

    # High-resolution fine-tuned 384x384 checkpoint.
    "caformer_m36.sail_in22k_ft_in1k_384",


    # =========================================================================
    # CAFormer-B36
    # Biggest/base variant (~99M params), highest accuracy but heavier compute.
    # =========================================================================

    # Base model trained on ImageNet-1K at 224x224.
    "caformer_b36.sail_in1k",

    # Higher-resolution 384x384 ImageNet-1K version.
    "caformer_b36.sail_in1k_384",

    # Pretrained on large-scale ImageNet-22K dataset.
    "caformer_b36.sail_in22k",

    # ImageNet-22K pretrained and ImageNet-1K fine-tuned checkpoint.
    "caformer_b36.sail_in22k_ft_in1k",

    # Best-performing high-resolution 384x384 fine-tuned variant.
    "caformer_b36.sail_in22k_ft_in1k_384",
]

## ACTIVE MODELS

In [ ]:
# ── Active models ─────────────────────────────────────────────────────────────
ACTIVE_MODELS = [
    "caformer_s18.sail_in22k_ft_in1k",
]


# ── Model registry ────────────────────────────────────────────────────────────
MODEL_REGISTRY: dict[str, dict] = {

    # ─────────────────────────────────────────────────────────────────────────
    # CAFormer-S18
    # Fastest + best efficiency setup for Kaggle T4
    # ─────────────────────────────────────────────────────────────────────────
    "caformer_s18.sail_in22k_ft_in1k": {

        # 224 is MUCH faster than 384
        "image_size": 224,

        # T4-friendly
        # should still fit comfortably with AMP
        "batch_size": 48 if DEVICE_TYPE == "cuda" else 8,

        # stable finetuning LR
        "lr": 5e-4,

        # light freezing helps stability early
        "freeze_stages": 1,

        "regularization": {

            # enough regularization without slowing convergence
            "weight_decay": 2e-4,

            # moderate smoothing
            "label_smoothing": 0.10,

            # strong enough for generalization
            "mixup_alpha": 0.4,
            "mixup_prob": 0.5,

            # lighter CutMix for synthetic attribution
            "cutmix_alpha": 1.0,
            "cutmix_prob": 0.2,

            # 0.30 was too aggressive for S18 at 224
            "dropout": 0.22,

            # stable clipping
            "grad_clip_norm": 0.8,
        },
    },
}

## IMAGENET NORMALIZE STATS

In [ ]:
# ── ImageNet normalization stats ──────────────────────────────────────────────
IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32,
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32,
)


# ── Tensor variants (future GPU-side transforms support) ─────────────────────
IMAGENET_MEAN_TENSOR = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
IMAGENET_STD_TENSOR  = torch.tensor(IMAGENET_STD).view(3, 1, 1)


# ── Pillow compatibility-safe interpolation ──────────────────────────────────
try:
    RESAMPLE = Image.Resampling.BICUBIC
except AttributeError:
    RESAMPLE = Image.BICUBIC

## LOGGING AND SEEDING

In [ ]:
# ── Logging ───────────────────────────────────────────────────────────────────
def setup_logging(
    log_dir: Path | None = None,
) -> logging.Logger:

    if log_dir is None:
        log_dir = LOG_DIR

    logger = logging.getLogger("pipeline")

    # prevent duplicate handlers
    if logger.handlers:
        return logger

    logger.setLevel(logging.INFO)
    logger.propagate = False

    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    # console handler
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)

    logger.addHandler(sh)

    # file handler
    log_path = (
        log_dir
        / f"run_{time.strftime('%Y%m%d_%H%M%S')}.log"
    )

    fh = logging.FileHandler(
        log_path,
        encoding="utf-8",
    )

    fh.setFormatter(fmt)

    logger.addHandler(fh)

    return logger


# global placeholder logger only
LOGGER = logging.getLogger("pipeline")


def initialize_logger_safely():

    global LOGGER

    LOGGER = setup_logging()

    LOGGER.info(
        f"DEVICE={DEVICE_TYPE} | "
        f"NUM_DEVICES={NUM_DEVICES}"
    )

    if DEVICE_TYPE == "cuda":

        LOGGER.info(
            f"GPU={torch.cuda.get_device_name(0)} | "
            f"VRAM={torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB"
        )

    LOGGER.info(
        f"epochs={NUM_EPOCHS} | "
        f"seed={SEED}"
    )


# ── Seeding ───────────────────────────────────────────────────────────────────
def set_seed(
    seed: int = SEED,
    deterministic: bool = False,
) -> None:

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)

    if deterministic:

        torch.backends.cudnn.deterministic = True

        torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int):

    worker_seed = SEED + worker_id

    np.random.seed(worker_seed)

    random.seed(worker_seed)

## UTILITY

In [ ]:
# ── Utility helpers ───────────────────────────────────────────────────────────

def is_master_process() -> bool:
    return True

def save_json(obj, path: Path) -> None:
    ensure_dir(path.parent)

    with path.open("w", encoding="utf-8") as fh:
        json.dump(
            obj,
            fh,
            indent=2,
            default=str,
        )


def gpu_synchronize() -> None:
    """
    Explicit CUDA synchronization.

    Use ONLY for:
    - benchmarking
    - precise timing
    - debugging async CUDA failures
    """
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def clear_gpu_memory() -> None:
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def get_cuda_memory_summary() -> str:
    if not torch.cuda.is_available():
        return "CUDA unavailable"

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    max_allocated = torch.cuda.max_memory_allocated() / 1024**3

    return (
        f"allocated={allocated:.2f}GB | "
        f"reserved={reserved:.2f}GB | "
        f"max_allocated={max_allocated:.2f}GB"
    )


def resolve_model_settings(
    model_name: str,
    mcfg: dict | None = None,
) -> dict:
    """
    Merge per-model settings with global defaults.
    """

    base = copy.deepcopy(
        MODEL_REGISTRY[model_name]
        if mcfg is None else mcfg
    )

    reg = copy.deepcopy(
        CONTROL_PANEL["regularization_defaults"]
    )

    reg.update(base.get("regularization", {}))

    base["regularization"] = reg

    return base


def build_weighting_value(meta: dict) -> float:
    """
    Value used for weighted ensembling.
    """

    if BLEND_WEIGHT_METRIC == "val_f1_macro":
        return float(
            meta.get("val_metrics", {}).get(
                "f1_macro",
                meta.get("selection_value", 0.0),
            )
        )

    if BLEND_WEIGHT_METRIC == "accuracy":
        return float(
            meta.get("val_metrics", {}).get(
                "accuracy",
                meta.get("selection_value", 0.0),
            )
        )

    return float(
        meta.get(
            "selection_value",
            meta.get("val_metrics", {}).get(
                "f1_macro",
                0.0,
            ),
        )
    )


def make_stacking_learner(name: str):
    name = name.lower().strip()

    if name == "logreg":
        return LogisticRegression(
            max_iter=3000,
            solver="lbfgs",
        )

    if name == "ridge":
        return RidgeClassifier()

    if name == "mlp":
        return MLPClassifier(
            hidden_layer_sizes=(256,),
            activation="relu",
            max_iter=500,
            random_state=SEED,
        )

    raise ValueError(
        f"Unknown STACKING_LEARNER={name}"
    )


def predict_proba_from_learner(
    learner,
    x: np.ndarray,
) -> np.ndarray:

    if hasattr(learner, "predict_proba"):
        return np.asarray(
            learner.predict_proba(x),
            dtype=np.float32,
        )

    scores = np.asarray(
        learner.decision_function(x),
        dtype=np.float32,
    )

    if scores.ndim == 1:
        scores = np.stack(
            [-scores, scores],
            axis=1,
        )

    scores = scores - scores.max(
        axis=1,
        keepdims=True,
    )

    probs = np.exp(scores)

    probs /= probs.sum(
        axis=1,
        keepdims=True,
    )

    return probs.astype(np.float32)

# STAGE 2 — DATA LOADING & VALIDATION

In [ ]:
def load_raw_csvs() -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Read metadata CSVs and resolve absolute image paths.
    """

    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)

    def _full_path(split_dir: Path, raw: str) -> str:
        p = Path(str(raw))

        if p.is_absolute() and p.exists():
            return str(p)

        return str(split_dir / p.name)

    train_df["full_path"] = train_df["path"].map(
        lambda r: _full_path(TRAIN_DIR, r)
    )

    test_df["full_path"] = test_df["path"].map(
        lambda r: _full_path(TEST_DIR, r)
    )

    return train_df, test_df


def validate_data(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> None:
    """
    Hard validation checks before training starts.
    """

    # LOGGER.info("=" * 60)
    # LOGGER.info("STAGE 2 — DATA LOADING & VALIDATION")
    # LOGGER.info("=" * 60)

    LOGGER.info(
        f"train_rows={len(train_df):,} | "
        f"test_rows={len(test_df):,}"
    )

    # ── Required columns ─────────────────────────────────────────────────────
    required_train_cols = {"ID", "path", "y"}
    required_test_cols = {"ID", "path"}

    missing_train = required_train_cols - set(train_df.columns)
    missing_test = required_test_cols - set(test_df.columns)

    if missing_train:
        raise ValueError(
            f"Missing train columns: {sorted(missing_train)}"
        )

    if missing_test:
        raise ValueError(
            f"Missing test columns: {sorted(missing_test)}"
        )

    # ── NaN checks ───────────────────────────────────────────────────────────
    if train_df[["ID", "path", "y"]].isnull().any().any():
        raise ValueError("NaNs detected in training metadata")

    if test_df[["ID", "path"]].isnull().any().any():
        raise ValueError("NaNs detected in test metadata")

    # ── Duplicate checks ─────────────────────────────────────────────────────
    train_dup_ids = train_df["ID"].duplicated().sum()
    test_dup_ids = test_df["ID"].duplicated().sum()

    if train_dup_ids:
        LOGGER.warning(
            f"Duplicate train IDs detected: {train_dup_ids}"
        )

    if test_dup_ids:
        LOGGER.warning(
            f"Duplicate test IDs detected: {test_dup_ids}"
        )

    # ── Label checks ─────────────────────────────────────────────────────────
    invalid_labels = train_df[
        ~train_df["y"].between(0, NUM_CLASSES - 1)
    ]

    if len(invalid_labels):
        raise ValueError(
            f"Invalid labels detected: "
            f"{sorted(invalid_labels['y'].unique().tolist())}"
        )

    counts = train_df["y"].value_counts().sort_index()

    LOGGER.info(
        "Class distribution:\n"
        f"{counts.to_string()}"
    )

    LOGGER.info(
        "Class ratios:\n"
        f"{(counts / counts.sum()).round(4).to_string()}"
    )

    if len(counts) != NUM_CLASSES:
        LOGGER.warning(
            f"Expected {NUM_CLASSES} classes; "
            f"found {len(counts)}"
        )

    # ── File existence ───────────────────────────────────────────────────────
    train_exists = train_df["full_path"].map(
        lambda p: Path(p).exists()
    )

    test_exists = test_df["full_path"].map(
        lambda p: Path(p).exists()
    )

    missing_train = int((~train_exists).sum())
    missing_test = int((~test_exists).sum())

    LOGGER.info(f"Missing train images: {missing_train}")
    LOGGER.info(f"Missing test images: {missing_test}")

    if missing_train:
        missing_examples = train_df.loc[
            ~train_exists,
            "full_path",
        ].head(5).tolist()

        raise FileNotFoundError(
            f"{missing_train} training images missing. "
            f"Examples: {missing_examples}"
        )

    if missing_test:
        missing_examples = test_df.loc[
            ~test_exists,
            "full_path",
        ].head(5).tolist()

        raise FileNotFoundError(
            f"{missing_test} test images missing. "
            f"Examples: {missing_examples}"
        )

    LOGGER.info("Data validation passed ✓")

# STAGE 3 — DATA PREPROCESSING

In [ ]:
def _normalize_np(arr: np.ndarray) -> np.ndarray:
    """uint8 HWC → float32 HWC, ImageNet-normalised."""
    arr = arr.astype(np.float32) / 255.0
    return (arr - IMAGENET_MEAN) / IMAGENET_STD


_image_stats_logged = False

def load_image(path: str, image_size: int) -> np.ndarray:
    """
    Load, resize, normalise one image.
    Returns float32 array of shape (C, H, W) — channels-first for PyTorch.
    """
    global _image_stats_logged
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")
            img = img.resize((image_size, image_size), RESAMPLE)
            arr = np.asarray(img, dtype=np.uint8)          # (H, W, 3)  uint8
    except Exception as exc:
        raise FileNotFoundError(f"Cannot load image: {path}") from exc

    arr = _normalize_np(arr)                                # (H, W, 3)  float32
    arr = arr.transpose(2, 0, 1)                            # (3, H, W)  ← PyTorch expects channels-first
    arr = np.ascontiguousarray(arr)

    if not _image_stats_logged:
        LOGGER.info(
            f"[image_check] shape={arr.shape} dtype={arr.dtype} "
            f"min={arr.min():.3f} max={arr.max():.3f}"
        )
        _image_stats_logged = True
    return arr


class ImageDataset(Dataset):
    """Minimal dataset: loads images on the fly (no caching to save RAM)."""

    def __init__(
        self,
        paths: Sequence[str],
        labels: Sequence[int] | None = None,
        image_size: int = 224,
    ):
        self.paths      = [str(p) for p in paths]
        self.labels     = None if labels is None else np.asarray(labels, dtype=np.int64)
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        img = load_image(self.paths[idx], self.image_size)  # (3, H, W) float32
        t   = torch.from_numpy(img)
        if self.labels is None:
            return t
        return t, int(self.labels[idx])


def make_loader(
    paths,
    labels=None,
    *,
    image_size: int = 224,
    batch_size: int = 32,
    shuffle: bool = False,
    sampler=None,
    drop_last: bool = False,
    fold_seed: int = SEED,
) -> DataLoader:
    ds = ImageDataset(paths, labels, image_size=image_size)
    g = torch.Generator()
    g.manual_seed(fold_seed)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        drop_last=drop_last,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
        worker_init_fn=seed_worker,
        generator=g,
    )


def build_folds(y: np.ndarray, n_folds: int, seed: int) -> dict[int, dict]:

    """
    Builds folds.

    If n_folds == 1:
        uses a single stratified 90/10 split.

    Else:
        uses StratifiedKFold.
    """

    meta = {}

    # ─────────────────────────────────────────────────────────────────────
    # Single-fold mode
    # Better than 50/50 pseudo-kfold split
    # ─────────────────────────────────────────────────────────────────────
    if n_folds == 1:

        indices = np.arange(len(y))

        tr, va = train_test_split(
            indices,
            test_size=0.10,
            stratify=y,
            random_state=seed,
        )

        meta[0] = {
            "fold_idx": 0,

            "train_indices": tr.tolist(),
            "val_indices": va.tolist(),

            "train_count": int(len(tr)),
            "val_count": int(len(va)),

            "train_class_counts":
                np.bincount(y[tr], minlength=NUM_CLASSES).tolist(),

            "val_class_counts":
                np.bincount(y[va], minlength=NUM_CLASSES).tolist(),
        }

        LOGGER.info(
            f"[single split] "
            f"train={len(tr)}  val={len(va)}  "
            f"val_cls={meta[0]['val_class_counts']}"
        )

        return meta

    # ─────────────────────────────────────────────────────────────────────
    # Normal K-Fold mode
    # ─────────────────────────────────────────────────────────────────────
    skf = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=seed,
    )

    for fold_idx, (tr, va) in enumerate(
        skf.split(np.zeros(len(y)), y)
    ):

        meta[fold_idx] = {
            "fold_idx": int(fold_idx),

            "train_indices": tr.tolist(),
            "val_indices": va.tolist(),

            "train_count": int(len(tr)),
            "val_count": int(len(va)),

            "train_class_counts":
                np.bincount(y[tr], minlength=NUM_CLASSES).tolist(),

            "val_class_counts":
                np.bincount(y[va], minlength=NUM_CLASSES).tolist(),
        }

        LOGGER.info(
            f"[fold {fold_idx}] "
            f"train={len(tr)}  val={len(va)}  "
            f"val_cls={meta[fold_idx]['val_class_counts']}"
        )

    return meta

# STAGE 4 — MODEL DEFINITION

## MODEL BUILDING

In [ ]:
class TimmModel(nn.Module):
    """
    timm backbone wrapper with:
    - optional stage freezing
    - dropout support
    - compile support
    - diagnostics
    """

    def __init__(
        self,
        model_name: str,
        num_classes: int,
        dropout: float = 0.0,
        freeze_stages: int = 0,
    ):
        super().__init__()

        self.model_name = model_name

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=num_classes,
            drop_rate=dropout,
        )

        if freeze_stages > 0:
            self._freeze_stages(freeze_stages)

        total_params = sum(
            p.numel()
            for p in self.parameters()
        )

        trainable_params = sum(
            p.numel()
            for p in self.parameters()
            if p.requires_grad
        )

        LOGGER.info(
            f"[model] {model_name} | "
            f"params={total_params:,} | "
            f"trainable={trainable_params:,} | "
            f"dropout={dropout:.3f} | "
            f"freeze_stages={freeze_stages}"
        )

    def _freeze_stages(self, freeze_stages: int):
        """
        Conservative CAFormer freezing.

        Freezes earliest backbone children only.
        """

        children = list(self.backbone.children())

        frozen = 0

        for idx, module in enumerate(children):

            if idx >= freeze_stages:
                break

            for param in module.parameters():
                param.requires_grad = False

            frozen += 1

        LOGGER.info(
            f"[freeze] froze {frozen} backbone stages"
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:

        return self.backbone(x)


# ── Model builder ─────────────────────────────────────────────────────────────
def build_model(
    model_name: str,
    num_classes: int = NUM_CLASSES,
) -> nn.Module:

    cfg = resolve_model_settings(model_name)

    reg_cfg = cfg["regularization"]

    dropout = float(
        reg_cfg.get("dropout", 0.0)
    )

    freeze_stages = int(
        cfg.get("freeze_stages", 0)
    )

    model = TimmModel(
        model_name=model_name,
        num_classes=num_classes,
        dropout=dropout,
        freeze_stages=freeze_stages,
    )

    # channels-last improves CUDA throughput
    # if DEVICE_TYPE == "cuda":
    #     model = model.to(
    #         memory_format=torch.channels_last
    #     )

    model = model.to(DEVICE)
    # if DEVICE_TYPE == "cuda" and NUM_DEVICES > 1:
    
    #     LOGGER.info(
    #         f"[multi_gpu] DataParallel enabled ({NUM_DEVICES} GPUs)"
    #     )
    #     model = nn.DataParallel(model)

    # optional torch.compile
    if USE_COMPILE:
        LOGGER.info(
            f"[compile] compiling {model_name}"
        )

        model = torch.compile(model)

    return model

# STAGE 5 — TRAINING

## ENCODING

In [ ]:
# ── Soft-label helpers ────────────────────────────────────────────────────────
def smooth_one_hot(
    labels: torch.Tensor,
    num_classes: int,
    smoothing: float = 0.0,
) -> torch.Tensor:
    """
    Convert integer labels to label-smoothed soft targets.

    Returns:
        Tensor of shape (N, num_classes)
    """

    if not (0.0 <= smoothing < 1.0):
        raise ValueError(
            f"Invalid label smoothing value: {smoothing}"
        )

    confidence = 1.0 - smoothing

    with torch.no_grad():

        soft_targets = torch.full(
            (labels.size(0), num_classes),
            fill_value=smoothing / max(num_classes - 1, 1),
            device=labels.device,
            dtype=torch.float32,
        )

        soft_targets.scatter_(
            1,
            labels.unsqueeze(1),
            confidence,
        )

    return soft_targets


## CROSS ENTROPY

In [ ]:
def cross_entropy_soft(
    logits: torch.Tensor,
    soft_targets: torch.Tensor,
) -> torch.Tensor:
    """
    Cross-entropy for soft labels.

    Compatible with:
    - MixUp
    - CutMix
    - label smoothing
    """

    soft_targets = soft_targets.to(
        dtype=logits.dtype
    )

    log_probs = F.log_softmax(
        logits,
        dim=-1,
    )

    loss = -(
        soft_targets * log_probs
    ).sum(dim=-1)

    return loss.mean()

## BATCH REGULARIZATION

In [ ]:
# ── Batch regularization ──────────────────────────────────────────────────────
def _randperm(
    batch_size: int,
    device: torch.device,
) -> torch.Tensor:
    """
    Device-native random permutation.
    """

    return torch.randperm(
        batch_size,
        device=device,
    )


def mixup(
    images: torch.Tensor,
    labels: torch.Tensor,
    alpha: float,
):
    """
    Standard MixUp for images + soft labels.
    """

    if alpha <= 0.0 or images.size(0) < 2:
        return images, labels

    lam = float(
        np.random.beta(alpha, alpha)
    )

    perm = _randperm(
        images.size(0),
        images.device,
    )

    lam_t = torch.tensor(
        lam,
        device=images.device,
        dtype=images.dtype,
    )

    mixed_images = (
        lam_t * images
        + (1.0 - lam_t) * images[perm]
    )

    mixed_labels = (
        lam * labels
        + (1.0 - lam) * labels[perm]
    )

    return mixed_images, mixed_labels


def cutmix(
    images: torch.Tensor,
    labels: torch.Tensor,
    alpha: float,
):
    """
    Standard CutMix for NCHW tensors.
    """

    if alpha <= 0.0 or images.size(0) < 2:
        return images, labels

    lam = float(
        np.random.beta(alpha, alpha)
    )

    perm = _randperm(
        images.size(0),
        images.device,
    )

    b, c, h, w = images.shape

    cut_ratio = math.sqrt(1.0 - lam)

    cut_w = int(w * cut_ratio)
    cut_h = int(h * cut_ratio)

    cx = torch.randint(
        0,
        w,
        (1,),
        device=images.device,
    ).item()

    cy = torch.randint(
        0,
        h,
        (1,),
        device=images.device,
    ).item()

    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)

    x2 = min(cx + cut_w // 2, w)
    y2 = min(cy + cut_h // 2, h)

    mixed_images = images.clone()

    mixed_images[:, :, y1:y2, x1:x2] = (
        images[perm, :, y1:y2, x1:x2]
    )

    area = max(
        1,
        (x2 - x1) * (y2 - y1),
    )

    lam_adjusted = 1.0 - (
        area / float(h * w)
    )

    mixed_labels = (
        lam_adjusted * labels
        + (1.0 - lam_adjusted) * labels[perm]
    )

    return mixed_images, mixed_labels


def apply_batch_regularization(
    images: torch.Tensor,
    soft_labels: torch.Tensor,
    reg: dict,
):
    """
    Apply MixUp / CutMix regularization.
    """

    cutmix_alpha = float(
        reg.get("cutmix_alpha", 0.0)
    )

    cutmix_prob = float(
        reg.get("cutmix_prob", 0.0)
    )

    mixup_alpha = float(
        reg.get("mixup_alpha", 0.0)
    )

    mixup_prob = float(
        reg.get("mixup_prob", 0.0)
    )

    r = random.random()

    if (
        cutmix_alpha > 0.0
        and r < cutmix_prob
    ):
        return cutmix(
            images,
            soft_labels,
            cutmix_alpha,
        )

    if (
        mixup_alpha > 0.0
        and r < mixup_prob
    ):
        return mixup(
            images,
            soft_labels,
            mixup_alpha,
        )

    return images, soft_labels

## PERFORMANCE METRICS UTILS

In [ ]:
# ── Metric helpers ────────────────────────────────────────────────────────────
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict:
    """
    Compute classification metrics safely.
    """

    if len(y_true) == 0:
        return {
            "accuracy": 0.0,
            "f1_macro": 0.0,
            "f1_weighted": 0.0,
            "per_class_f1": [0.0] * NUM_CLASSES,
        }

    per_class_f1 = f1_score(
        y_true,
        y_pred,
        average=None,
        zero_division=0,
    )

    return {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),

        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "f1_weighted": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),

        "per_class_f1": per_class_f1.tolist(),
    }


# ── Generalization-aware checkpoint scoring ──────────────────────────────────
def generalization_score(
    train_m: dict,
    val_m: dict,
    *,
    cfg: dict | None = None,
    return_parts: bool = False,
):
    """
    Generalization-aware validation score.

    Important:
    With strong augmentation (MixUp/CutMix/RandAugment),
    train metrics naturally become lower and noisier.

    Therefore:
    - validation F1 remains primary signal
    - overfit penalties should remain moderate
    """

    gcfg = (
        CONTROL_PANEL["generalization"]
        if cfg is None
        else cfg
    )

    train_f1 = float(
        train_m.get("f1_macro", 0.0)
    )

    val_f1 = float(
        val_m.get("f1_macro", 0.0)
    )

    val_weight = float(
        gcfg.get("val_weight", 1.0)
    )

    low_train_reward = float(
        gcfg.get("low_train_reward", 0.0)
    )

    overfit_penalty = float(
        gcfg.get("overfit_penalty", 1.0)
    )

    balance_penalty = float(
        gcfg.get("balance_penalty", 0.0)
    )

    positive_gap = max(
        0.0,
        val_f1 - train_f1,
    )

    negative_gap = max(
        0.0,
        train_f1 - val_f1,
    )

    balance_gap = abs(
        train_f1 - val_f1
    )

    # primary objective = strong validation F1
    score = (
        val_weight * val_f1

        # slight reward when val > train
        + low_train_reward * positive_gap

        # penalize obvious overfit
        - overfit_penalty * (negative_gap ** 2)

        # encourage train/val consistency
        - balance_penalty * (balance_gap ** 2)
    )

    parts = {
        "train_f1": float(train_f1),
        "val_f1": float(val_f1),

        "positive_gap": float(positive_gap),
        "negative_gap": float(negative_gap),
        "balance_gap": float(balance_gap),

        "val_term": float(
            val_weight * val_f1
        ),

        "low_train_reward": float(
            low_train_reward * positive_gap
        ),

        "overfit_penalty_term": float(
            overfit_penalty * (negative_gap ** 2)
        ),

        "balance_penalty_term": float(
            balance_penalty * (balance_gap ** 2)
        ),

        "score": float(score),
    }

    if return_parts:
        return float(score), parts

    return float(score)


# ── Checkpoint-selection metric resolver ─────────────────────────────────────
def get_selection_value(
    train_m: dict,
    val_m: dict,
) -> float:
    """
    Resolve checkpoint-selection metric.
    """

    metric = (
        CHECKPOINT_SELECTION_METRIC
        .strip()
        .lower()
    )

    if metric == "val_accuracy":
        return float(
            val_m.get("accuracy", 0.0)
        )

    if metric == "val_f1_macro":
        return float(
            val_m.get("f1_macro", 0.0)
        )

    if metric == "generalization_score":
        return float(
            val_m.get(
                "generalization_score",
                val_m.get("f1_macro", 0.0),
            )
        )

    return float(
        val_m.get(
            metric,
            val_m.get("f1_macro", 0.0),
        )
    )

## RESOURCE UTILS

In [ ]:
# ── Resource helpers ──────────────────────────────────────────────────────────
def _disk_gib() -> float:
    """
    Current working-directory disk usage in GiB.
    """

    total_bytes = sum(
        f.stat().st_size
        for f in WORKING_ROOT.rglob("*")
        if f.is_file()
    )

    return total_bytes / (1024 ** 3)


def _ram_gib() -> float:
    """
    Current system RAM usage in GiB.
    """

    return (
        psutil.virtual_memory().used
        / (1024 ** 3)
    )


def _gpu_memory_gib() -> dict:
    """
    CUDA memory summary.

    Returns:
        {
            "allocated": ...,
            "reserved": ...,
            "max_allocated": ...
        }
    """

    if not torch.cuda.is_available():
        return {
            "allocated": 0.0,
            "reserved": 0.0,
            "max_allocated": 0.0,
        }

    return {
        "allocated": (
            torch.cuda.memory_allocated()
            / (1024 ** 3)
        ),

        "reserved": (
            torch.cuda.memory_reserved()
            / (1024 ** 3)
        ),

        "max_allocated": (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        ),
    }


def _elapsed_h() -> float:
    """
    Session elapsed time in hours.
    """

    return (
        time.time() - SESSION_START_TIME
    ) / 3600.0


def _remaining_h() -> float:
    """
    Remaining session budget in hours.
    """

    return (
        SESSION_BUDGET_SECS
        - (time.time() - SESSION_START_TIME)
    ) / 3600.0


def log_resources(tag: str = "") -> None:
    """
    Log runtime resource usage and enforce limits.
    """

    disk = _disk_gib()

    ram = _ram_gib()

    gpu = _gpu_memory_gib()

    remaining = max(
        0.0,
        _remaining_h(),
    )

    msg = (
        f"[resources{' ' + tag if tag else ''}] "
        f"disk={disk:.2f}GiB | "
        f"ram={ram:.1f}GiB | "
        f"elapsed={_elapsed_h():.2f}h | "
        f"remaining={remaining:.2f}h"
    )

    if torch.cuda.is_available():

        msg += (
            f" | "
            f"cuda_alloc={gpu['allocated']:.2f}GiB"
            f" | "
            f"cuda_reserved={gpu['reserved']:.2f}GiB"
            f" | "
            f"cuda_peak={gpu['max_allocated']:.2f}GiB"
        )

    LOGGER.info(msg)

    # ── Hard disk limit ──────────────────────────────────────────────────────
    if disk > DISK_LIMIT_GIB:

        raise RuntimeError(
            f"DISK LIMIT EXCEEDED: "
            f"{disk:.2f} > {DISK_LIMIT_GIB:.2f} GiB"
        )

    # ── RAM warning ──────────────────────────────────────────────────────────
    if ram > RAM_WARN_GIB:

        LOGGER.warning(
            f"[ram] High RAM usage: "
            f"{ram:.1f} GiB"
        )

    # ── CUDA fragmentation warning ───────────────────────────────────────────
    if torch.cuda.is_available():

        allocated = gpu["allocated"]
        reserved = gpu["reserved"]

        if (
            reserved > 0.1
            and allocated / reserved < 0.6
        ):
            LOGGER.warning(
                "[cuda] High memory fragmentation detected "
                f"(allocated={allocated:.2f}GiB "
                f"reserved={reserved:.2f}GiB)"
            )


def budget_ok(
    reserve_minutes: float = 5.0,
) -> bool:
    """
    Check whether enough session budget remains.

    reserve_minutes:
        Safety margin to avoid abrupt Kaggle shutdowns.
    """

    reserve_secs = reserve_minutes * 60.0

    elapsed = (
        time.time() - SESSION_START_TIME
    )

    return elapsed < (
        SESSION_BUDGET_SECS - reserve_secs
    )

## CHECKPOINT SAVE UTILS

In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────
def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer=None,
    scheduler=None,
    scaler=None,
    meta: dict | None = None,
) -> None:
    """
    Save training checkpoint safely.
    """

    ensure_dir(path.parent)

    # support compiled models
    state_model = (
        model._orig_mod
        if hasattr(model, "_orig_mod")
        else model
    )

    checkpoint = {
        "model_state_dict": state_model.state_dict(),
        "metadata": meta or {},
    }

    if optimizer is not None:
        checkpoint["optimizer_state_dict"] = (
            optimizer.state_dict()
        )

    if scheduler is not None:
        checkpoint["scheduler_state_dict"] = (
            scheduler.state_dict()
        )

    if scaler is not None:
        checkpoint["scaler_state_dict"] = (
            scaler.state_dict()
        )

    tmp_path = path.with_suffix(".tmp")

    torch.save(
        checkpoint,
        tmp_path,
    )

    tmp_path.replace(path)

    save_json(
        meta or {},
        path.with_suffix(".json"),
    )


def load_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer=None,
    scheduler=None,
    scaler=None,
    strict: bool = True,
):
    """
    Load checkpoint safely.
    """

    LOGGER.info(
        f"[checkpoint] loading {path}"
    )

    try:

        ckpt = torch.load(
            path,
            map_location="cpu",
        )

    except Exception as exc:

        raise RuntimeError(
            f"Failed to load checkpoint: {path}"
        ) from exc

    state_model = (
        model._orig_mod
        if hasattr(model, "_orig_mod")
        else model
    )

    state_model.load_state_dict(
        ckpt["model_state_dict"],
        strict=strict,
    )

    if (
        optimizer is not None
        and "optimizer_state_dict" in ckpt
    ):
        optimizer.load_state_dict(
            ckpt["optimizer_state_dict"]
        )

    if (
        scheduler is not None
        and "scheduler_state_dict" in ckpt
    ):
        scheduler.load_state_dict(
            ckpt["scheduler_state_dict"]
        )

    if (
        scaler is not None
        and "scaler_state_dict" in ckpt
    ):
        scaler.load_state_dict(
            ckpt["scaler_state_dict"]
        )

    meta = ckpt.get("metadata", {})

    LOGGER.info(
        f"[checkpoint] loaded {path.name}"
    )

    return model, meta


def prune_checkpoints(
    saved: list,
    keep: int,
) -> list:
    """
    Keep strongest checkpoints only.
    """

    if keep <= 0:
        return []

    saved = sorted(
        saved,
        key=lambda r: r["sv"],
        reverse=True,
    )

    to_remove = saved[keep:]

    for item in to_remove:

        p = Path(item["path"])

        try:

            if p.exists():
                p.unlink()

            jf = p.with_suffix(".json")

            if jf.exists():
                jf.unlink()

            LOGGER.info(
                f"[prune] removed {p.name} "
                f"sv={item['sv']:.4f}"
            )

        except Exception as exc:

            LOGGER.warning(
                f"[prune] failed removing "
                f"{p}: {exc}"
            )

    return saved[:keep]

## EVAL AND PREDICT LOADERS

In [ ]:
# ── Per-split predictions ─────────────────────────────────────────────────────
@torch.inference_mode()
def predict_loader(
    model: nn.Module,
    loader: DataLoader,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:

    model.eval()

    all_probs = []
    all_true = []

    for batch in loader:

        has_labels = (
            isinstance(batch, (list, tuple))
            and len(batch) == 2
        )

        if has_labels:
            imgs, lbls = batch

            all_true.append(
                lbls.numpy()
            )

        else:
            imgs = batch

        imgs = imgs.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=USE_AMP,
        ):

            logits = model(imgs)

            probs = F.softmax(
                logits,
                dim=-1,
            )

        all_probs.append(
            probs.float().cpu().numpy()
        )

        del imgs, logits, probs

    probs_arr = (
        np.concatenate(all_probs, axis=0)
        if all_probs
        else np.empty(
            (0, NUM_CLASSES),
            dtype=np.float32,
        )
    )

    preds_arr = (
        probs_arr.argmax(axis=1)
        if len(probs_arr)
        else np.array([], dtype=np.int64)
    )

    true_arr = (
        np.concatenate(all_true, axis=0)
        if all_true
        else np.array([], dtype=np.int64)
    )

    return (
        true_arr,
        preds_arr,
        probs_arr,
    )


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.inference_mode()
def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    criterion=None,
) -> dict:
    """
    Full evaluation pass.
    """

    model.eval()

    all_probs = []
    all_true = []

    total_loss = 0.0
    total_count = 0

    for batch in loader:

        has_labels = (
            isinstance(batch, (list, tuple))
            and len(batch) == 2
        )

        if has_labels:

            imgs, lbls = batch

            all_true.append(
                lbls.numpy()
            )

            lbls_dev = lbls.to(
                DEVICE,
                non_blocking=True,
            )

        else:

            imgs = batch

            lbls_dev = None

        imgs = imgs.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=USE_AMP,
        ):

            logits = model(imgs)

            probs = F.softmax(
                logits,
                dim=-1,
            )

            # clean validation CE
            if (
                has_labels
                and criterion is not None
            ):

                loss = F.cross_entropy(
                    logits,
                    lbls_dev,
                )

        all_probs.append(
            probs.float().cpu().numpy()
        )

        if (
            has_labels
            and criterion is not None
        ):

            bs = int(imgs.size(0))

            total_loss += (
                float(loss.item()) * bs
            )

            total_count += bs

        del imgs, logits, probs

        if has_labels:
            del lbls, lbls_dev

    probs_arr = (
        np.concatenate(all_probs, axis=0)
        if all_probs
        else np.empty(
            (0, NUM_CLASSES),
            dtype=np.float32,
        )
    )

    preds_arr = (
        probs_arr.argmax(axis=1)
        if len(probs_arr)
        else np.array([], dtype=np.int64)
    )

    true_arr = (
        np.concatenate(all_true, axis=0)
        if all_true
        else np.array([], dtype=np.int64)
    )

    metrics = compute_metrics(
        true_arr,
        preds_arr,
    )

    metrics["loss"] = (
        float(total_loss / max(total_count, 1))
        if total_count
        else None
    )

    metrics["probs"] = probs_arr

    metrics["y_true"] = true_arr

    metrics["y_pred"] = preds_arr

    return metrics

## CORE TRAINING LOOP

In [ ]:
# ── Core training loop for one fold ───────────────────────────────────────────
def train_fold(
    fold_idx: int,
    fold_info: dict,
    train_df: pd.DataFrame,
    model_name: str,
    mcfg: dict,
    arch_ckpt_dir: Path,
) -> list[dict]:
    
    set_seed(SEED + fold_idx)
    
    cfg = resolve_model_settings(
        model_name,
        mcfg,
    )

    reg = cfg["regularization"]

    LOGGER.info(
        f"[fold {fold_idx}] START | model={model_name} | "
        f"img={cfg['image_size']} | bs={cfg['batch_size']} | lr={cfg['lr']:.1e} | "
        f"wd={cfg['regularization']['weight_decay']} | "
        f"train={fold_info['train_count']} | val={fold_info['val_count']}"
    )

    # ── Config ────────────────────────────────────────────────────────────────
    img_sz = int(cfg["image_size"])

    batch_size = int(
        cfg["batch_size"]
    )

    lr = float(
        cfg["lr"]
    )

    weight_decay = float(
        reg.get("weight_decay", 0.0)
    )

    label_smoothing = float(
        reg.get("label_smoothing", 0.0)
    )

    grad_clip_norm = float(
        reg.get("grad_clip_norm", 1.0)
    )

    grad_accum_steps = max(
        1,
        int(GRAD_ACCUM_STEPS),
    )

    fold_dir = ensure_dir(
        arch_ckpt_dir / f"fold_{fold_idx}"
    )

    # ── Split data ────────────────────────────────────────────────────────────
    tr_rows = train_df.iloc[
        fold_info["train_indices"]
    ].reset_index(drop=True)

    va_rows = train_df.iloc[
        fold_info["val_indices"]
    ].reset_index(drop=True)

    x_tr = tr_rows["full_path"] \
        .astype(str) \
        .to_numpy()

    y_tr = tr_rows["y"] \
        .astype(np.int64) \
        .to_numpy()

    x_va = va_rows["full_path"] \
        .astype(str) \
        .to_numpy()

    y_va = va_rows["y"] \
        .astype(np.int64) \
        .to_numpy()


    # ── DataLoaders ───────────────────────────────────────────────────────────
    train_loader = make_loader(
        x_tr,
        y_tr,
        image_size=img_sz,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        fold_seed=SEED + fold_idx,
    )

    val_loader = make_loader(
        x_va,
        y_va,
        image_size=img_sz,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        fold_seed=SEED + fold_idx,
    )

    train_eval_loader = None

    if TRAIN_EVAL_EACH_EPOCH:

        train_eval_loader = make_loader(
            x_tr,
            y_tr,
            image_size=img_sz,
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            fold_seed=SEED + fold_idx,
        )

    # ── Model ────────────────────────────────────────────────────────────────
    model = build_model(model_name)
    if PRINT_MODEL:
        print(model)

    trainable_params = [
        p
        for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = None

    if USE_LR_PLATEAU:

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=PLATEAU_FACTOR,
            patience=PLATEAU_PATIENCE,
            min_lr=PLATEAU_MIN_LR,
        )

    criterion = cross_entropy_soft

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )

    # ── Training state ───────────────────────────────────────────────────────
    history: list[dict] = []

    saved_ckpts: list[dict] = []

    best_sel_value = -float("inf")

    no_improve_cnt = 0

    # ── Epoch loop ────────────────────────────────────────────────────────────
    for epoch in range(NUM_EPOCHS):

        if not budget_ok():

            LOGGER.info(
                "[budget] "
                "Time limit reached — stopping training."
            )

            break

        epoch_start = time.time()

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        running_loss = 0.0

        running_correct = 0

        running_seen = 0

        grad_norms = []

        processed_images = 0

        all_train_preds = []
        all_train_labels = []

        # ── Batch loop ───────────────────────────────────────────────────────
        for step, (imgs, lbls) in enumerate(train_loader):

            imgs = imgs.to(
                DEVICE,
                non_blocking=True,
            ).contiguous(memory_format=torch.channels_last)

            lbls = lbls.to(
                DEVICE,
                non_blocking=True,
            )

            # ── Soft targets ────────────────────────────────────────────────
            soft_targets = smooth_one_hot(
                lbls,
                NUM_CLASSES,
                label_smoothing,
            )

            imgs, soft_targets = apply_batch_regularization(
                imgs,
                soft_targets,
                reg,
            )

            # ── Forward ─────────────────────────────────────────────────────
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=AMP_DTYPE,
                enabled=USE_AMP,
            ):

                logits = model(imgs)

                loss = criterion(
                    logits,
                    soft_targets,
                )

                loss = (
                    loss
                    / grad_accum_steps
                )

            # ── Backward ────────────────────────────────────────────────────
            scaler.scale(loss).backward()

            do_step = (
                (step + 1) % grad_accum_steps == 0
                or (step + 1) == len(train_loader)
            )

            if do_step:

                scaler.unscale_(optimizer)

                grad_norm = float(
                    torch.nn.utils.clip_grad_norm_(
                        trainable_params,
                        max_norm=grad_clip_norm,
                    ).item()
                )

                grad_norms.append(
                    grad_norm
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

            # ── Metrics ─────────────────────────────────────────────────────
            with torch.no_grad():

                preds = logits.argmax(
                    dim=1
                )

                correct = int(
                    (preds == lbls)
                    .sum()
                    .item()
                )

            all_train_preds.append(preds.cpu().numpy())
            all_train_labels.append(lbls.cpu().numpy())
            
            bs = int(
                imgs.size(0)
            )

            processed_images += bs

            running_seen += bs

            running_correct += correct

            running_loss += (
                float(loss.item())
                * bs
                * grad_accum_steps
            )

            loss_val = float(
                loss.item()
                * grad_accum_steps
            )

            if not np.isfinite(loss_val):

                LOGGER.warning(
                    f"[train] "
                    f"non-finite loss "
                    f"epoch={epoch} "
                    f"step={step} "
                    f"loss={loss_val}"
                )

            # ── Verbose logging ────────────────────────────────────────────
            if (
                VERBOSE_TRAINING
                and step % 50 == 0
            ):

                train_acc = (
                    running_correct
                    / max(running_seen, 1)
                )

                LOGGER.info(
                    f"[train] "
                    f"epoch={epoch:02d} "
                    f"step={step:04d} "
                    f"loss={loss_val:.4f} "
                    f"acc={train_acc:.4f} "
                    f"lr={optimizer.param_groups[0]['lr']:.2e}"
                )

                if (
                    torch.cuda.is_available()
                    and PRINT_CUDA_MEMORY_EVERY > 0
                    and step % PRINT_CUDA_MEMORY_EVERY == 0
                ):

                    allocated = (
                        torch.cuda.memory_allocated()
                        / 1024**2
                    )

                    reserved = (
                        torch.cuda.memory_reserved()
                        / 1024**2
                    )

                    LOGGER.info(
                        f"[cuda_memory] "
                        f"allocated={allocated:.1f}MiB "
                        f"reserved={reserved:.1f}MiB"
                    )

            del (
                imgs,
                lbls,
                soft_targets,
                logits,
                preds,
                loss,
            )

        # ── Epoch metrics ────────────────────────────────────────────────────
        train_loss = (
            running_loss
            / max(running_seen, 1)
        )

        train_acc = (
            running_correct
            / max(running_seen, 1)
        )

        train_eval = None

        if (
            TRAIN_EVAL_EACH_EPOCH
            and train_eval_loader is not None
        ):

            train_eval = evaluate_loader(
                model,
                train_eval_loader,
                criterion=criterion,
            )

        val_eval = evaluate_loader(
            model,
            val_loader,
            criterion=criterion,
        )

        # ── Build metric dicts ──────────────────────────────────────────────
        if train_eval is not None:

            train_m = train_eval.copy()

        else:

            train_m = compute_metrics(
                np.concatenate(all_train_labels),
                np.concatenate(all_train_preds),
            )
            train_m["accuracy"] = float(train_acc)
            train_m["loss"] = float(train_loss)

        val_m = val_eval.copy()

        # remove heavy arrays
        for d in (train_m, val_m):

            d.pop("probs", None)

            d.pop("y_true", None)

            d.pop("y_pred", None)

        train_m["loss"] = float(
            train_loss
        )

        # ── Generalization scoring ──────────────────────────────────────────
        val_m["generalization_score"], gen_parts = (
            generalization_score(
                train_m,
                val_m,
                return_parts=True,
            )
        )

        selection_value = get_selection_value(
            train_m,
            val_m,
        )

        # ── Worst classes ───────────────────────────────────────────────────
        per_class_f1 = val_eval.get(
            "per_class_f1",
            [0.0] * NUM_CLASSES,
        )

        worst3 = sorted(
            enumerate(per_class_f1),
            key=lambda t: t[1],
        )[:3]

        # ── Epoch summary ───────────────────────────────────────────────────
        epoch_time = (
            time.time()
            - epoch_start
        )

        imgs_per_sec = (
            processed_images
            / max(epoch_time, 1e-6)
        )

        train_gap = (
            train_m["f1_macro"]
            - val_m["f1_macro"]
        )

        LOGGER.info(
            f"\n{'─'*100}\n"
            f"Epoch {epoch:02d}/{NUM_EPOCHS - 1} | "
            f"fold={fold_idx} | "
            f"model={model_name}\n\n"

            f"TRAIN "
            f"loss={train_m['loss']:.4f} | "
            f"acc={train_m['accuracy']:.4f} | "
            f"f1={train_m['f1_macro']:.4f}\n"

            f"VAL   "
            f"loss={val_m['loss']:.4f} | "
            f"acc={val_m['accuracy']:.4f} | "
            f"f1={val_m['f1_macro']:.4f}\n\n"

            f"gap={train_gap:+.4f} | "
            f"gen_score={val_m['generalization_score']:.4f} | "
            f"selection={selection_value:.4f}\n\n"

            f"throughput={imgs_per_sec:.0f} img/s | "
            f"lr={optimizer.param_groups[0]['lr']:.2e} | "
            f"grad_clip={grad_clip_norm:.2f}\n\n"

            f"worst_classes={worst3}\n"
            f"{'─'*100}"
        )

        # ── Checkpoint ──────────────────────────────────────────────────────
        ckpt_path = (
            fold_dir
            / f"epoch_{epoch:03d}.pt"
        )

        meta = {
            "model_name": model_name,
            "fold_idx": fold_idx,
            "epoch": epoch,

            "image_size": img_sz,
            "batch_size": batch_size,

            "train_metrics": train_m,
            "val_metrics": val_m,

            "selection_metric": CHECKPOINT_SELECTION_METRIC,
            "selection_value": float(selection_value),

            "generalization_parts": gen_parts,

            "regularization": reg,
        }

        save_checkpoint(
            ckpt_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            meta=meta,
        )

        saved_ckpts.append({
            "path": str(ckpt_path),
            "sv": float(selection_value),
        })

        saved_ckpts = prune_checkpoints(
            saved_ckpts,
            CHECKPOINT_KEEP_TOP_K,
        )

        history.append({
            "epoch": epoch,
            "train": train_m,
            "val": val_m,
            "sv": float(selection_value),
            "generalization_parts": gen_parts,
        })

        # ── Scheduler ───────────────────────────────────────────────────────
        if scheduler is not None:

            scheduler.step(
                selection_value
            )

        # ── Best checkpoint ─────────────────────────────────────────────────
        if selection_value > best_sel_value + 1e-4:

            best_sel_value = (
                selection_value
            )

            no_improve_cnt = 0

            best_dst = ensure_dir(
                FINAL_DIR / model_name
            ) / f"fold_{fold_idx}_best.pt"

            shutil.copy2(
                ckpt_path,
                best_dst,
            )

            shutil.copy2(
                ckpt_path.with_suffix(".json"),
                best_dst.with_suffix(".json"),
            )

            LOGGER.info(
                f"[best] "
                f"fold={fold_idx} "
                f"epoch={epoch} "
                f"selection_value={best_sel_value:.4f} "
                f"→ {best_dst.name}"
            )

        else:

            no_improve_cnt += 1

            LOGGER.info(
                f"[early_stop] "
                f"no improvement "
                f"{no_improve_cnt}/{EARLY_STOP_PATIENCE}"
            )

            if no_improve_cnt >= EARLY_STOP_PATIENCE:

                LOGGER.info(
                    f"[early_stop] "
                    f"triggered at epoch={epoch}"
                )

                break

        # ── Resources ───────────────────────────────────────────────────────
        log_resources(
            f"fold={fold_idx} epoch={epoch}"
        )

    # ── Save history ─────────────────────────────────────────────────────────
    save_json(
        history,
        LOG_DIR / (
            f"{model_name[:30]}"
            f"_fold{fold_idx}_history.json"
        ),
    )

    # ── Cleanup ──────────────────────────────────────────────────────────────
    del (
        model,
        optimizer,
        scheduler,
        scaler,
    )

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

    gc.collect()

    log_resources(
        f"after fold {fold_idx}"
    )

    return history

# STAGE 6 — INFERENCE / TEST-SET PREDICTION

## FUNCTIONS

In [ ]:
# ── Checkpoint loading ────────────────────────────────────────────────────────
def load_model_from_ckpt(
    ckpt_path: Path,
) -> tuple[nn.Module, dict, str]:

    LOGGER.info(
        f"[load_ckpt] loading {ckpt_path}"
    )

    ckpt = torch.load(
        ckpt_path,
        map_location="cpu",
    )

    meta = ckpt.get(
        "metadata",
        {},
    )

    model_name = meta.get(
        "model_name",
        ACTIVE_MODELS[0],
    )

    val_m = meta.get(
        "val_metrics",
        {},
    )

    LOGGER.info(
        f"[load_ckpt] "
        f"{ckpt_path.name} | "
        f"arch={model_name} | "
        f"epoch={meta.get('epoch', '?')} | "
        f"val_f1={val_m.get('f1_macro', float('nan')):.4f} | "
        f"gen={val_m.get('generalization_score', float('nan')):.4f}"
    )

    model = build_model(
        model_name
    )

    state_dict = ckpt[
        "model_state_dict"
    ]

    missing_keys, unexpected_keys = model.load_state_dict(
        state_dict,
        strict=False,
    )

    if missing_keys:

        LOGGER.warning(
            f"[load_ckpt] missing_keys={missing_keys}"
        )

    if unexpected_keys:

        LOGGER.warning(
            f"[load_ckpt] unexpected_keys={unexpected_keys}"
        )

    model.eval()

    return (
        model,
        meta,
        model_name,
    )


# ── Single-pass inference ────────────────────────────────────────────────────
@torch.inference_mode()
def infer_single(
    model: nn.Module,
    loader: DataLoader,
) -> np.ndarray:

    model.eval()

    probs = []

    for batch in loader:

        imgs = (
            batch[0]
            if isinstance(batch, (list, tuple))
            else batch
        )

        imgs = imgs.to(
            DEVICE,
            non_blocking=True,
        )

        if DEVICE_TYPE == "cuda":

            imgs = imgs.contiguous(
                memory_format=torch.channels_last
            )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=USE_AMP,
        ):

            logits = model(imgs)

            p = F.softmax(
                logits,
                dim=-1,
            )

        probs.append(
            p.float().cpu().numpy()
        )

        del imgs, logits, p

    return (
        np.concatenate(probs, axis=0)
        if probs
        else np.empty(
            (0, NUM_CLASSES),
            dtype=np.float32,
        )
    )


# ── Test-time augmentation inference ─────────────────────────────────────────
@torch.inference_mode()
def infer_tta(
    model: nn.Module,
    paths,
    img_sz: int,
    batch_size: int,
    n: int,
) -> np.ndarray:

    loader = make_loader(
        paths,
        image_size=img_sz,
        batch_size=batch_size,
        shuffle=False,
    )

    accum_probs = None

    total_passes = n + 1

    for t in range(total_passes):

        pass_probs = []

        for batch in loader:

            imgs = (
                batch[0]
                if isinstance(batch, (list, tuple))
                else batch
            )

            imgs = imgs.to(
                DEVICE,
                non_blocking=True,
            )

            if DEVICE_TYPE == "cuda":

                imgs = imgs.contiguous(
                    memory_format=torch.channels_last
                )

            # ── GPU-side TTA ───────────────────────────────────────────────
            if t > 0:

                # horizontal flip
                if t % 2 == 1:

                    imgs = torch.flip(
                        imgs,
                        dims=[3],
                    )

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=AMP_DTYPE,
                enabled=USE_AMP,
            ):

                logits = model(imgs)

                probs = F.softmax(
                    logits,
                    dim=-1,
                )

            pass_probs.append(
                probs.float().cpu().numpy()
            )

            del imgs, logits, probs

        pass_probs = np.concatenate(
            pass_probs,
            axis=0,
        )

        if accum_probs is None:

            accum_probs = pass_probs

        else:

            accum_probs += pass_probs

        LOGGER.info(
            f"[TTA] pass {t + 1}/{total_passes}"
        )

    return (
        accum_probs
        / float(total_passes)
    )

## CORE INFERENCE LOOP

In [ ]:
# ── Ensemble inference / blending / stacking ─────────────────────────────────
def run_ensemble_inference(
    train_meta: pd.DataFrame,
    test_meta: pd.DataFrame,
    y_train: np.ndarray,
    fold_metadata: dict,
) -> tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    pd.DataFrame,
    int,
] | None:

    LOGGER.info("=" * 96)

    LOGGER.info(
        "STAGE 6 — INFERENCE / TEST-SET PREDICTION"
    )

    LOGGER.info("=" * 96)

    LOGGER.info(
        f"[blend] "
        f"mode={BLEND_MODE} | "
        f"weight_metric={BLEND_WEIGHT_METRIC} | "
        f"stacking_learner={STACKING_LEARNER}"
    )

    if test_meta.empty:

        LOGGER.warning(
            "test_meta is empty — skipping inference."
        )

        return None

    # ── Discover checkpoints ────────────────────────────────────────────────
    if RUN_INFERENCE_ONLY:

        ckpt_paths = sorted(
            Path(INFERENCE_ONLY_PATH).glob(
                "**/*best.pt"
            )
        )

    else:

        ckpt_paths = sorted(
            FINAL_DIR.rglob(
                "fold_*_best.pt"
            )
        )

    LOGGER.info(
        f"[inference] "
        f"found {len(ckpt_paths)} checkpoint(s)"
    )

    if not ckpt_paths:

        LOGGER.error(
            "No checkpoints found."
        )

        return None

    test_paths = (
        test_meta["full_path"]
        .astype(str)
        .to_numpy()
    )

    # ── Averaging ensembles ─────────────────────────────────────────────────
    if BLEND_MODE in {
        "simple_avg",
        "weighted_avg",
    }:

        all_probs: list[np.ndarray] = []

        all_weights: list[float] = []

        for idx, ckpt_path in enumerate(ckpt_paths):

            model, meta, model_name = (
                load_model_from_ckpt(
                    ckpt_path
                )
            )

            img_sz = int(
                meta.get(
                    "image_size",
                    MODEL_REGISTRY[
                        model_name
                    ]["image_size"],
                )
            )

            batch_size = int(
                meta.get(
                    "batch_size",
                    MODEL_REGISTRY[
                        model_name
                    ]["batch_size"],
                )
            )

            t0 = time.time()

            # ── Inference ────────────────────────────────────────────────
            if USE_TTA:

                probs = infer_tta(
                    model=model,
                    paths=test_paths,
                    img_sz=img_sz,
                    batch_size=batch_size,
                    n=TTA_N,
                )

            else:

                loader = make_loader(
                    test_paths,
                    image_size=img_sz,
                    batch_size=batch_size,
                    shuffle=False,
                    drop_last=False,
                )

                probs = infer_single(
                    model,
                    loader,
                )

            elapsed = max(
                time.time() - t0,
                1e-6,
            )

            speed = (
                len(test_paths)
                / elapsed
            )

            LOGGER.info(
                f"[predict] "
                f"ckpt={idx} | "
                f"model={model_name} | "
                f"shape={probs.shape} | "
                f"speed={speed:.1f} img/s"
            )

            all_probs.append(
                probs.astype(np.float32)
            )

            weight = max(
                float(
                    build_weighting_value(
                        meta
                    )
                ),
                1e-6,
            )

            all_weights.append(
                weight
            )

            LOGGER.info(
                f"[weight] "
                f"value={weight:.6f} | "
                f"selection={meta.get('selection_value', float('nan')):.6f}"
            )

            del model

            clear_gpu_memory()

        # ── Blend ─────────────────────────────────────────────────────────
        stacked = np.stack(
            all_probs,
            axis=0,
        )

        weights = np.asarray(
            all_weights,
            dtype=np.float32,
        )

        weights = np.nan_to_num(
            weights,
            nan=1e-6,
            posinf=1e-6,
            neginf=1e-6,
        )

        if BLEND_MODE == "simple_avg":

            ensemble = stacked.mean(
                axis=0
            )

            LOGGER.info(
                "[ensemble] simple_avg applied"
            )

        else:

            weights = (
                weights
                / max(weights.sum(), 1e-6)
            )

            LOGGER.info(
                f"[ensemble] "
                f"weighted_avg weights="
                f"{weights.round(4).tolist()}"
            )

            ensemble = np.average(
                stacked,
                axis=0,
                weights=weights,
            )

        ensemble = ensemble.astype(
            np.float32
        )

        preds = ensemble.argmax(
            axis=1
        ).astype(np.int64)

        conf = ensemble.max(
            axis=1
        ).astype(np.float32)

        LOGGER.info(
            f"[ensemble] "
            f"models={len(all_probs)} | "
            f"mean_conf={conf.mean():.4f} | "
            f"min={conf.min():.4f} | "
            f"max={conf.max():.4f}"
        )

        return (
            ensemble,
            preds,
            conf,
            test_meta,
            len(ckpt_paths),
        )

    # ── Stacking ensemble ───────────────────────────────────────────────────
    model_ckpts: dict[
        str,
        list[Path],
    ] = {}

    for model_name in ACTIVE_MODELS:

        model_ckpts[model_name] = sorted(
            (
                FINAL_DIR
                / model_name
            ).glob(
                "fold_*_best.pt"
            )
        )

        LOGGER.info(
            f"[stacking] "
            f"model={model_name} | "
            f"ckpts={len(model_ckpts[model_name])}"
        )

    n_train = len(
        train_meta
    )

    n_classes = NUM_CLASSES

    train_blocks: list[np.ndarray] = []

    test_blocks: list[np.ndarray] = []

    # ── Per-model OOF/test feature generation ───────────────────────────────
    for model_name, ckpts in model_ckpts.items():

        if not ckpts:

            raise RuntimeError(
                f"No checkpoints for stacking: "
                f"{model_name}"
            )

        model_oof = np.zeros(
            (n_train, n_classes),
            dtype=np.float32,
        )

        model_test_fold_probs = []

        for ckpt_path in ckpts:

            model, meta, loaded_name = (
                load_model_from_ckpt(
                    ckpt_path
                )
            )

            fold_idx = int(
                meta.get(
                    "fold_idx",
                    -1,
                )
            )

            if fold_idx not in fold_metadata:

                raise RuntimeError(
                    f"Fold metadata missing "
                    f"for fold_idx={fold_idx}"
                )

            img_sz = int(
                meta.get(
                    "image_size",
                    MODEL_REGISTRY[
                        loaded_name
                    ]["image_size"],
                )
            )

            batch_size = int(
                meta.get(
                    "batch_size",
                    MODEL_REGISTRY[
                        loaded_name
                    ]["batch_size"],
                )
            )

            # ── Validation OOF prediction ────────────────────────────────
            va_idx = fold_metadata[
                fold_idx
            ]["val_indices"]

            va_rows = train_meta.iloc[
                va_idx
            ].reset_index(drop=True)

            va_loader = make_loader(
                va_rows["full_path"]
                .astype(str)
                .to_numpy(),

                va_rows["y"]
                .astype(np.int64)
                .to_numpy(),

                image_size=img_sz,
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )

            _, _, va_probs = predict_loader(
                model,
                va_loader,
            )

            model_oof[
                va_idx
            ] = va_probs.astype(
                np.float32
            )

            LOGGER.info(
                f"[stacking] "
                f"OOF filled | "
                f"model={model_name} | "
                f"fold={fold_idx} | "
                f"shape={va_probs.shape}"
            )

            # ── Test prediction ──────────────────────────────────────────
            test_loader = make_loader(
                test_paths,
                image_size=img_sz,
                batch_size=batch_size,
                shuffle=False,
                drop_last=False,
            )

            _, _, te_probs = predict_loader(
                model,
                test_loader,
            )

            model_test_fold_probs.append(
                te_probs.astype(np.float32)
            )

            del model

            clear_gpu_memory()

        # ── Average fold predictions ─────────────────────────────────────
        model_test_avg = np.mean(
            np.stack(
                model_test_fold_probs,
                axis=0,
            ),
            axis=0,
        ).astype(np.float32)

        train_blocks.append(
            model_oof
        )

        test_blocks.append(
            model_test_avg
        )

        LOGGER.info(
            f"[stacking] "
            f"model={model_name} | "
            f"train_block={model_oof.shape} | "
            f"test_block={model_test_avg.shape}"
        )

    # ── Build meta-features ──────────────────────────────────────────────────
    X_train = np.concatenate(
        train_blocks,
        axis=1,
    )

    X_test = np.concatenate(
        test_blocks,
        axis=1,
    )

    LOGGER.info(
        f"[stacking] "
        f"X_train={X_train.shape} | "
        f"X_test={X_test.shape}"
    )

    # ── Train stacker ────────────────────────────────────────────────────────
    stacker = make_stacking_learner(
        STACKING_LEARNER
    )

    LOGGER.info(
        f"[stacking] "
        f"fitting learner={STACKING_LEARNER}"
    )

    stacker.fit(
        X_train,
        y_train,
    )

    ensemble = predict_proba_from_learner(
        stacker,
        X_test,
    ).astype(np.float32)

    preds = ensemble.argmax(
        axis=1
    ).astype(np.int64)

    conf = ensemble.max(
        axis=1
    ).astype(np.float32)

    LOGGER.info(
        f"[stacking] "
        f"learner={STACKING_LEARNER} | "
        f"models={len(model_ckpts)} | "
        f"mean_conf={conf.mean():.4f} | "
        f"min={conf.min():.4f} | "
        f"max={conf.max():.4f}"
    )

    return (
        ensemble,
        preds,
        conf,
        test_meta,
        len(ckpt_paths),
    )

# STAGE 7 — SUBMISSION & OUTPUTS

In [ ]:
# ── Submission & output generation ────────────────────────────────────────────
def write_submission(
    test_meta: pd.DataFrame,
    preds: np.ndarray,
    conf: np.ndarray,
    num_models: int,
) -> pd.DataFrame:

    LOGGER.info("=" * 96)

    LOGGER.info(
        "STAGE 7 — SUBMISSION & OUTPUTS"
    )

    LOGGER.info("=" * 96)

    # ── Validation ───────────────────────────────────────────────────────────
    if len(test_meta) != len(preds):

        raise ValueError(
            f"Prediction length mismatch: "
            f"len(test_meta)={len(test_meta)} "
            f"vs len(preds)={len(preds)}"
        )

    if len(conf) != len(preds):

        raise ValueError(
            f"Confidence length mismatch: "
            f"len(conf)={len(conf)} "
            f"vs len(preds)={len(preds)}"
        )

    if len(preds) == 0:

        raise RuntimeError(
            "No predictions generated."
        )

    preds = np.asarray(
        preds,
        dtype=np.int64,
    )

    conf = np.asarray(
        conf,
        dtype=np.float32,
    )

    # ── Build submission ─────────────────────────────────────────────────────
    submission = test_meta[
        ["ID"]
    ].copy()

    submission["TARGET"] = preds

    # ── Output paths ─────────────────────────────────────────────────────────
    submission_root_path = (
        WORKING_ROOT
        / "submission.csv"
    )

    submission_infer_path = (
        INFERENCE_DIR
        / "submission.csv"
    )

    confidence_path = (
        INFERENCE_DIR
        / "prediction_confidence.csv"
    )

    metadata_path = (
        INFERENCE_DIR
        / "submission_metadata.json"
    )

    # ── Save submission ──────────────────────────────────────────────────────
    submission.to_csv(
        submission_root_path,
        index=False,
    )

    submission.to_csv(
        submission_infer_path,
        index=False,
    )

    # ── Confidence outputs ───────────────────────────────────────────────────
    confidence_df = pd.DataFrame({
        "ID": test_meta["ID"].values,
        "predicted_class": preds,
        "confidence": conf,
    })

    confidence_df.to_csv(
        confidence_path,
        index=False,
    )

    # ── Prediction distribution ──────────────────────────────────────────────
    pred_distribution = (
        pd.Series(preds)
        .value_counts()
        .sort_index()
    )

    # ── Metadata ─────────────────────────────────────────────────────────────
    metadata = {

        "timestamp": pd.Timestamp.now(
            "UTC"
        ).isoformat(),

        "num_models": int(
            num_models
        ),

        "blend_mode": str(
            BLEND_MODE
        ),

        "tta_enabled": bool(
            USE_TTA
        ),

        "tta_passes": int(
            TTA_N if USE_TTA else 0
        ),

        "num_predictions": int(
            len(preds)
        ),

        "mean_confidence": float(
            conf.mean()
        ),

        "std_confidence": float(
            conf.std()
        ),

        "min_confidence": float(
            conf.min()
        ),

        "max_confidence": float(
            conf.max()
        ),

        "prediction_distribution": {
            str(k): int(v)
            for k, v in pred_distribution.items()
        },

        "device_type": DEVICE_TYPE,

        "amp_enabled": bool(
            USE_AMP
        ),

        "active_models": list(
            ACTIVE_MODELS
        ),
    }

    save_json(
        metadata,
        metadata_path,
    )

    # ── Logging ──────────────────────────────────────────────────────────────
    LOGGER.info(
        f"\n{'=' * 48}\n"
        f"Submission preview:\n"
        f"{submission.head(10).to_string(index=False)}"
    )

    LOGGER.info(
        f"[submission] "
        f"saved={submission_root_path}"
    )

    LOGGER.info(
        f"[confidence] "
        f"saved={confidence_path}"
    )

    LOGGER.info(
        f"[metadata] "
        f"saved={metadata_path}"
    )

    LOGGER.info(
        f"Prediction distribution:\n"
        f"{pred_distribution.to_string()}"
    )

    LOGGER.info(
        f"Confidence stats | "
        f"mean={conf.mean():.4f} | "
        f"std={conf.std():.4f} | "
        f"min={conf.min():.4f} | "
        f"max={conf.max():.4f}"
    )

    return submission

# MAIN ENTRY POINT

In [ ]:
# ── Main pipeline entry ───────────────────────────────────────────────────────
def run_pipeline(
    rank: int = 0,
) -> None:

    # ── Logger init ──────────────────────────────────────────────────────────
    initialize_logger_safely()

    LOGGER.info("=" * 100)

    LOGGER.info(
        "PIPELINE START"
    )

    LOGGER.info("=" * 100)

    # ── Reproducibility ──────────────────────────────────────────────────────
    set_seed(
        SEED
    )

    if is_master_process():

        log_resources(
            "startup"
        )

        LOGGER.info(
            f"[runtime] "
            f"device={DEVICE_TYPE} | "
            f"amp={USE_AMP} | "
            f"compile={USE_COMPILE} | "
            f"grad_accum={GRAD_ACCUM_STEPS}"
        )

        if torch.cuda.is_available():

            LOGGER.info(
                f"[gpu] "
                f"name={torch.cuda.get_device_name(0)}"
            )

    # ── STAGE 2: Load & validate ────────────────────────────────────────────
    LOGGER.info("=" * 96)

    LOGGER.info(
        "STAGE 2 — DATA LOADING & VALIDATION"
    )

    LOGGER.info("=" * 96)

    train_df, test_df = load_raw_csvs()

    validate_data(
        train_df,
        test_df,
    )

    # ── STAGE 3: Fold generation ────────────────────────────────────────────
    LOGGER.info("=" * 96)

    LOGGER.info(
        "STAGE 3 — DATA PREPROCESSING"
    )

    LOGGER.info("=" * 96)

    y_train = train_df["y"].to_numpy(
        dtype=np.int64
    )

    fold_metadata = build_folds(
        y_train,
        NUM_FOLDS,
        SEED,
    )

    if is_master_process():

        save_json(
            fold_metadata,
            PROCESSED_DIR
            / "fold_metadata.json",
        )

    # ── Inference-only mode ─────────────────────────────────────────────────
    if RUN_INFERENCE_ONLY:

        LOGGER.info("=" * 96)

        LOGGER.info(
            "RUN MODE — INFERENCE ONLY"
        )

        LOGGER.info("=" * 96)

        result = run_ensemble_inference(
            train_meta=train_df,
            test_meta=test_df,
            y_train=y_train,
            fold_metadata=fold_metadata,
        )

        if result is not None:

            (
                _,
                preds,
                conf,
                test_meta,
                num_models,
            ) = result

            if is_master_process():

                write_submission(
                    test_meta=test_meta,
                    preds=preds,
                    conf=conf,
                    num_models=num_models,
                )

        LOGGER.info(
            "[pipeline] inference-only completed"
        )

        return

    # ── Training mode ───────────────────────────────────────────────────────
    if RUN_TRAINING:

        LOGGER.info("=" * 96)

        LOGGER.info(
            "RUN MODE — TRAINING + INFERENCE"
        )

        LOGGER.info("=" * 96)

        all_history: dict = {}

        # ── Per-model training ─────────────────────────────────────────────
        for model_name in ACTIVE_MODELS:

            if not budget_ok():

                LOGGER.warning(
                    "[budget] session budget exhausted"
                )

                break


            mcfg = MODEL_REGISTRY[
                model_name
            ]

            arch_dir = (
                CHECKPOINT_DIR
                / model_name
            )

            model_hist = {}

            # ── Fold loop ────────────────────────────────────────────────
            for fold_idx, fold_info in fold_metadata.items():

                if not budget_ok():

                    LOGGER.warning(
                        "[budget] "
                        "session budget exhausted"
                    )

                    break

                fold_history = train_fold(
                    fold_idx=fold_idx,
                    fold_info=fold_info,
                    train_df=train_df,
                    model_name=model_name,
                    mcfg=mcfg,
                    arch_ckpt_dir=arch_dir,
                )

                model_hist[
                    fold_idx
                ] = fold_history

                clear_gpu_memory()

            all_history[
                model_name
            ] = model_hist

            # ── Save per-model history ────────────────────────────────
            if is_master_process():

                save_json(
                    model_hist,
                    LOG_DIR / (
                        f"{model_name[:30]}"
                        f"_all_folds.json"
                    ),
                )

        # ── Save full history ───────────────────────────────────────────────
        if is_master_process():

            save_json(
                all_history,
                LOG_DIR
                / "full_training_history.json",
            )

        # ── STAGE 6: Ensemble inference ─────────────────────────────────────
        result = run_ensemble_inference(
            train_meta=train_df,
            test_meta=test_df,
            y_train=y_train,
            fold_metadata=fold_metadata,
        )

        # ── STAGE 7: Submission generation ─────────────────────────────────
        if result is not None:

            (
                _,
                preds,
                conf,
                test_meta,
                num_models,
            ) = result

            if is_master_process():

                write_submission(
                    test_meta=test_meta,
                    preds=preds,
                    conf=conf,
                    num_models=num_models,
                )

        # ── Final resource logging ─────────────────────────────────────────
        if is_master_process():

            log_resources(
                "final"
            )

        LOGGER.info(
            "[pipeline] training pipeline completed"
        )

        return

    # ── Invalid runtime config ──────────────────────────────────────────────
    raise RuntimeError(
        "Invalid runtime configuration. "
        "Set RUN_TRAINING=True "
        "or RUN_INFERENCE_ONLY=True."
    )


# ── Entrypoint ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    run_pipeline()